# Tutorial: Causal Machine Learning and the Resource Curse (EconML)

**Double Machine Learning Causal Forest with Simulated Data**

Based on Hodler, Lechner & Raschky (2023) "Institutions and the resource curse: New insights from causal machine learning" (*PLoS ONE*)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/starter-academic-v501/blob/master/content/post/python_EconML/references/tutorial-econml-resource-curse.ipynb)

---

## Abstract

This tutorial teaches the **EconML Causal Forest** (`CausalForestDML`) estimator through a simulated dataset that reproduces the key findings of Hodler, Lechner & Raschky (2023). EconML implements causal forests via the **Double Machine Learning** (DML) framework. Using simulated data with known ground-truth effects, we show how the method detects (i) positive mining effects, (ii) non-linear price effects, and (iii) institutional moderation of mining but not price effects. Runtime: ~5 minutes.

## 1. Setup and Installation

In [ ]:
# Install dependencies (Colab only)
!pip install -q econml matplotlib pandas numpy

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import econml
print(f"econml version: {econml.__version__}")

# Site color palette — dark theme
STEEL_BLUE = "#6a9bcc"
WARM_ORANGE = "#d97757"
NEAR_BLACK = "#141413"
TEAL = "#00d4c8"
DARK_NAVY = "#0f1729"
GRID_LINE = "#1f2b5e"
LIGHT_TEXT = "#c8d0e0"
WHITE_TEXT = "#e8ecf2"

plt.rcParams.update({
    "figure.facecolor": DARK_NAVY,
    "axes.facecolor": DARK_NAVY,
    "axes.edgecolor": DARK_NAVY,
    "axes.linewidth": 0,
    "axes.labelcolor": LIGHT_TEXT,
    "axes.titlecolor": WHITE_TEXT,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": False,
    "axes.spines.bottom": False,
    "axes.grid": True,
    "grid.color": GRID_LINE,
    "grid.linewidth": 0.6,
    "grid.alpha": 0.8,
    "xtick.color": LIGHT_TEXT,
    "ytick.color": LIGHT_TEXT,
    "xtick.major.size": 0,
    "ytick.major.size": 0,
    "text.color": WHITE_TEXT,
    "font.size": 12,
    "legend.frameon": False,
    "legend.fontsize": 11,
    "legend.labelcolor": LIGHT_TEXT,
    "figure.edgecolor": DARK_NAVY,
    "savefig.facecolor": DARK_NAVY,
    "savefig.edgecolor": DARK_NAVY,
})

### Ground-Truth Parameters

The simulated data embeds three findings with known causal parameters:

In [ ]:
# Ground-truth parameters embedded in the DGP
TRUE_PARAMS = {
    'ntl_mining_base': 0.25,         # Mining effect at mean institutions
    'ntl_mining_inst_mod': 0.15,     # Institutional moderation of mining
    'ntl_price_med': 0.05,           # Medium price premium (small)
    'ntl_price_high': 0.30,          # High price premium (large)
    'ntl_noise_sd': 0.25,            # Outcome noise
    'conflict_mining_base': 0.70,    # Mining increases conflict
    'conflict_mining_inst_mod': -0.50,  # Institutions dampen mining-conflict
    'conflict_price_med': 0.15,
    'conflict_price_high': 0.50,
    'conflict_base_rate': 0.12,
}

def expected_ates_ntl():
    p = TRUE_PARAMS
    return {
        '1-0': p['ntl_mining_base'],
        '2-0': p['ntl_mining_base'] + p['ntl_price_med'],
        '3-0': p['ntl_mining_base'] + p['ntl_price_high'],
        '2-1': p['ntl_price_med'],
        '3-1': p['ntl_price_high'],
        '3-2': p['ntl_price_high'] - p['ntl_price_med'],
    }

print("Ground-truth NTL ATEs:")
for comp, val in expected_ates_ntl().items():
    print(f"  ATE({comp}) = {val:.3f}")

## 2. Load Simulated Data

The simulated dataset (3,000 observations, 300 districts, 8 countries) uses known ground-truth causal effects embedded in the data-generating process.

In [ ]:
# Load from GitHub (works in Colab without local files)
DATA_URL = ("https://github.com/quarcs-lab/data-open"
            "/raw/master/stata19/sim_resource_curse.csv")

df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df):,} observations")
print(f"Districts: {df['district_id'].nunique()}, "
      f"Countries: {df['country_id'].nunique()}, "
      f"Years: {df['year'].min()}-{df['year'].max()}")
print(f"Mining districts: {df.loc[df['mining']==1, 'district_id'].nunique()} "
      f"({df['mining'].mean():.0%})")

## 3. Descriptive Statistics

In [ ]:
labels = {0: 'No mining', 1: 'Low prices', 2: 'Med prices', 3: 'High prices'}

# Treatment distribution chart
fig, ax = plt.subplots(figsize=(8, 3))
fig.patch.set_linewidth(0)
colors = [LIGHT_TEXT, STEEL_BLUE, TEAL, WARM_ORANGE]
counts = df['treatment'].value_counts().sort_index()
bars = ax.barh([labels[t] for t in counts.index], counts.values,
               color=colors, edgecolor=DARK_NAVY, linewidth=0.8)
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{count:,} ({count/len(df):.1%})', va='center',
            fontsize=9, color=LIGHT_TEXT)
ax.set_xlabel('Number of observations')
ax.set_title('Treatment Distribution (M=4)')
plt.tight_layout()
plt.show()

In [ ]:
# Outcomes by treatment group
print(f"{'Treatment':<20s} {'Mean NTL':>10s} {'Conflict Rate':>15s}")
print("-" * 47)
for t in sorted(df['treatment'].unique()):
    mask = df['treatment'] == t
    m_ntl = df.loc[mask, 'ntl_log'].mean()
    m_conf = df.loc[mask, 'conflict'].mean()
    print(f"{t} ({labels[t]:<13s})  {m_ntl:>10.3f} {m_conf:>14.1%}")

### Naive Comparison: Why We Need Causal ML

Simple difference-in-means is **biased** because mining districts differ systematically from non-mining districts. The DML Causal Forest corrects for this by residualizing both outcomes and treatments.

In [ ]:
gt = expected_ates_ntl()

print("Simple difference-in-means (no confounder adjustment):")
print(f"{'Comparison':<15s} {'Naive':>8s} {'Ground Truth':>14s} {'Bias':>8s}")
print("-" * 47)

for comp in ['1-0', '2-1', '3-1']:
    a, b = int(comp[0]), int(comp[2])
    mean_a = df.loc[df['treatment'] == a, 'ntl_log'].mean()
    mean_b = df.loc[df['treatment'] == b, 'ntl_log'].mean()
    naive = mean_a - mean_b
    truth = gt[comp]
    bias = naive - truth
    print(f"{comp:<15s} {naive:>8.3f} {truth:>14.3f} {bias:>+8.3f}")

## 4. EconML Causal Forest Estimation

### How the DML Causal Forest Works

1. **First stage**: Estimate nuisance models E[Y|X,W] and E[T|X,W] using Gradient Boosting
2. **Residualize**: Compute residuals that remove confounding variation
3. **Second stage**: Fit an honest causal forest on the residuals to estimate CATEs

**Neyman orthogonality**: First-stage estimation errors have only second-order impact on the causal estimates, making DML substantially more robust than naive two-stage procedures.

**Key design choice**: X features (heterogeneity) vs W features (controls only)
- **X**: Variables the causal forest can split on to discover heterogeneity
- **W**: Additional controls used only in the first-stage nuisance models

**Causal identification**: Requires the Conditional Independence Assumption (CIA) --- treatment is independent of potential outcomes conditional on (X, W). In this simulation, this holds by construction. In real data, unobserved confounders could bias estimates.

**Panel data**: `groups=district_id` ensures cross-fitting does not split within districts (prevents data leakage). Note: this does NOT provide clustered standard errors.

In [ ]:
from econml.dml import CausalForestDML
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier

# Heterogeneity features (X): causal forest splits on these
X_COLS = [
    'exec_constraints', 'quality_of_govt', 'gdp_pc',
    'elevation', 'temperature', 'ruggedness',
    'distance_capital', 'agri_suitability', 'population', 'ethnic_frac',
]

# Additional controls (W): first-stage only
W_COLS = ['country_id', 'year']

def fit_causal_forest(df, outcome, label):
    """Fit CausalForestDML for one outcome."""
    Y = df[outcome].values
    T = df['treatment'].values
    X = df[X_COLS].values
    W = df[W_COLS].values

    est = CausalForestDML(
        model_y=GradientBoostingRegressor(
            n_estimators=200, max_depth=4, random_state=42),
        model_t=GradientBoostingClassifier(
            n_estimators=200, max_depth=4, random_state=42),
        discrete_treatment=True,
        categories=[0, 1, 2, 3],
        n_estimators=500,
        min_samples_leaf=10,
        max_depth=None,
        honest=True,
        inference=True,
        cv=5,
        n_jobs=1,
        random_state=42,
    )

    t0 = time.time()
    est.fit(Y, T, X=X, W=W, groups=df['district_id'].values)
    elapsed = time.time() - t0
    print(f"  {label}: fitted in {elapsed:.0f}s")
    return est

In [ ]:
# Estimate NTL effects
est_ntl = fit_causal_forest(df, 'ntl_log', 'NTL')

In [ ]:
# Estimate Conflict effects
est_conf = fit_causal_forest(df, 'conflict', 'Conflict')

## 5. Average Treatment Effects (cf. Table 2)

EconML's `ate_inference()` provides ATEs with proper confidence intervals via the Bootstrap of Little Bags (BLB) method.

In [ ]:
X = df[X_COLS].values
gt = expected_ates_ntl()

comparisons = [
    ('1-0', 0, 1), ('2-0', 0, 2), ('3-0', 0, 3),
    ('2-1', 1, 2), ('3-1', 1, 3), ('3-2', 2, 3),
]

print(f"{'Comp':<8s} {'NTL Effect':>11s} {'NTL SE':>8s} "
      f"{'90% CI':>20s} {'Ground Truth':>13s} {'Conflict':>10s}")
print("-" * 72)

for comp_label, t0, t1 in comparisons:
    ntl_res = est_ntl.ate_inference(X, T0=t0, T1=t1)
    ntl_lo, ntl_hi = ntl_res.conf_int_mean(alpha=0.1)  # 90% CI
    conf_res = est_conf.ate_inference(X, T0=t0, T1=t1)

    sig = ''
    if ntl_res.stderr_mean > 0:
        z = abs(ntl_res.mean_point / ntl_res.stderr_mean)
        if z > 2.576: sig = '***'
        elif z > 1.96: sig = '**'
        elif z > 1.645: sig = '*'

    print(f"{comp_label:<8s} {ntl_res.mean_point:>8.4f}{sig:<3s} "
          f"{ntl_res.stderr_mean:>8.4f} "
          f"{'[' + f'{ntl_lo:.3f}, {ntl_hi:.3f}' + ']':>20s} "
          f"{gt.get(comp_label, np.nan):>13.3f} "
          f"{conf_res.mean_point:>10.4f}")

print("\nKey findings:")
print("  Finding 1: All mining-vs-no-mining effects are positive")
print("  Finding 2: ATE(2-1) is small/n.s., ATE(3-1) is large/significant")
print("\n  * p<0.10, ** p<0.05, *** p<0.01")

## 6. Treatment Effect Heterogeneity: GATEs (cf. Figures 2-3)

Unlike MCF, EconML does not compute GATEs automatically. We compute them manually:
1. Estimate individual CATEs via `est.effect(X, T0, T1)`
2. Group by institutional variable
3. Compute mean CATE per group = GATE

In [ ]:
def compute_gate(est, df, z_var, t0, t1):
    """Compute GATEs by grouping individual effects.

    Uses effect_inference() for proper BLB standard errors that capture
    estimation uncertainty, not just within-group heterogeneity.
    """
    X = df[X_COLS].values
    inf = est.effect_inference(X, T0=t0, T1=t1)
    ite = inf.point_estimate
    ite_se = inf.stderr

    z_vals = sorted(df[z_var].unique())
    gate_data = []
    for z in z_vals:
        mask = df[z_var].values == z
        n = mask.sum()
        gate_mean = ite[mask].mean()
        # SE of group mean: propagate per-observation BLB standard errors
        gate_se = np.sqrt(np.mean(ite_se[mask] ** 2) / n)
        gate_data.append({
            'z_value': z, 'gate': gate_mean, 'se': gate_se,
            'lower': gate_mean - 1.645 * gate_se,
            'upper': gate_mean + 1.645 * gate_se,
            'n': n,
        })
    return pd.DataFrame(gate_data), ite

### GATEs by Executive Constraints (cf. Paper Figure 2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_linewidth(0)

panels = [
    (est_ntl, 0, 1, 'NTL: Mining vs No Mining (1-0)', axes[0, 0], STEEL_BLUE),
    (est_ntl, 1, 3, 'NTL: High vs Low Prices (3-1)', axes[0, 1], STEEL_BLUE),
    (est_conf, 0, 1, 'Conflict: Mining vs No Mining (1-0)', axes[1, 0], TEAL),
    (est_conf, 1, 3, 'Conflict: High vs Low Prices (3-1)', axes[1, 1], TEAL),
]

for est, t0, t1, title, ax, color in panels:
    gate_df, ite = compute_gate(est, df, 'exec_constraints', t0, t1)

    ax.fill_between(gate_df['z_value'], gate_df['lower'],
                    gate_df['upper'], alpha=0.25, color=color)
    ax.plot(gate_df['z_value'], gate_df['gate'], 'o-',
            color=WHITE_TEXT, markersize=6, linewidth=1.5,
            markeredgecolor=DARK_NAVY, markeredgewidth=0.8, zorder=3)
    ate_val = ite.mean()
    ax.axhline(ate_val, color=WARM_ORANGE, linewidth=1.5, linestyle='--',
               alpha=0.8, label=f'ATE = {ate_val:.3f}')
    ax.set_xlabel('Constraints on the Executive')
    ax.set_ylabel('GATE')
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.suptitle('GATEs by Executive Constraints (EconML CausalForestDML)',
             fontsize=14, fontweight='bold', color=WHITE_TEXT, y=1.02)
plt.tight_layout()
plt.show()

### GATEs by Quality of Government (cf. Paper Figure 3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_linewidth(0)

panels = [
    (est_ntl, 0, 1, 'NTL: Mining vs No Mining (1-0)', axes[0, 0], STEEL_BLUE),
    (est_ntl, 1, 3, 'NTL: High vs Low Prices (3-1)', axes[0, 1], STEEL_BLUE),
    (est_conf, 0, 1, 'Conflict: Mining vs No Mining (1-0)', axes[1, 0], TEAL),
    (est_conf, 1, 3, 'Conflict: High vs Low Prices (3-1)', axes[1, 1], TEAL),
]

for est, t0, t1, title, ax, color in panels:
    gate_df, ite = compute_gate(est, df, 'quality_of_govt', t0, t1)

    ax.fill_between(gate_df['z_value'], gate_df['lower'],
                    gate_df['upper'], alpha=0.25, color=color)
    ax.plot(gate_df['z_value'], gate_df['gate'], 'o-',
            color=WHITE_TEXT, markersize=4, linewidth=1.5,
            markeredgecolor=DARK_NAVY, markeredgewidth=0.8, zorder=3)
    ate_val = ite.mean()
    ax.axhline(ate_val, color=WARM_ORANGE, linewidth=1.5, linestyle='--',
               alpha=0.8, label=f'ATE = {ate_val:.3f}')
    ax.set_xlabel('Quality of Government (ICRG)')
    ax.set_ylabel('GATE')
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.suptitle('GATEs by Quality of Government (EconML CausalForestDML)',
             fontsize=14, fontweight='bold', color=WHITE_TEXT, y=1.02)
plt.tight_layout()
plt.show()

### Finding 3: The Key Insight

**Left panels (1-0):** Mining effects should vary with institutions (upward slope for NTL)

**Right panels (3-1):** Price effects should be **flat** across institutions

Institutions moderate *mining* effects but NOT *price* effects.

In [ ]:
# GATE values table
print("GATE values for NTL by Executive Constraints:")
print("=" * 65)

for (t0, t1), comp_label in [((0, 1), 'Mining vs No Mining (1-0)'),
                               ((1, 3), 'High vs Low Prices (3-1)')]:
    gate_df, _ = compute_gate(est_ntl, df, 'exec_constraints', t0, t1)
    print(f"\n  {t1}-{t0} ({comp_label}):")
    print(f"  {'Exec. Constr.':<15s} {'GATE':>8s} {'90% CI':>20s} {'N':>6s}")
    print(f"  {'-'*52}")
    for _, row in gate_df.iterrows():
        print(f"  {row['z_value']:>13.0f}   {row['gate']:>8.3f}   "
              f"[{row['lower']:.3f}, {row['upper']:.3f}] {row['n']:>6.0f}")
    rng = gate_df['gate'].max() - gate_df['gate'].min()
    print(f"  Range: {rng:.3f}")

print("\n  1-0: effects INCREASE with institutions (range > 0)")
print("  3-1: effects are FLAT across institutions (range ~ 0)")

## 7. Variable Importance

In [ ]:
importances = est_ntl.feature_importances_
vim_data = sorted(zip(X_COLS, importances), key=lambda x: x[1], reverse=True)

print("Feature importances (heterogeneity drivers):")
for var, imp in vim_data:
    bar = '#' * int(imp * 100)
    print(f"  {var:<25s} {imp:>6.3f}  {bar}")

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_linewidth(0)
vars_, imps = zip(*reversed(vim_data))
ax.barh(vars_, imps, color=STEEL_BLUE, edgecolor=DARK_NAVY,
        linewidth=0.8, alpha=0.9)
ax.set_xlabel('Feature Importance (heterogeneity contribution)')
ax.set_title('Treatment Effect Heterogeneity Drivers (NTL)',
             fontweight='bold')
plt.tight_layout()
plt.show()

print("\nNote: These measure which features drive treatment effect")
print("HETEROGENEITY, not outcome prediction.")

## 8. CATE Interpreter (EconML Bonus Feature)

EconML's `SingleTreeCateInterpreter` fits a shallow decision tree to the estimated CATEs, creating interpretable subgroups. This is not available in the MCF package.

In [ ]:
from econml.cate_interpreter import SingleTreeCateInterpreter

intrp = SingleTreeCateInterpreter(max_depth=2, min_samples_leaf=100)
intrp.interpret(est_ntl, df[X_COLS].values)

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_linewidth(0)
# Tree nodes are hard to read on dark backgrounds — use a light inset
ax.set_facecolor('#f8f9fa')
intrp.plot(feature_names=X_COLS, ax=ax)
plt.title('Interpretable CATE Subgroups: Mining vs No Mining (1-0)',
          fontweight='bold', color=NEAR_BLACK)
plt.tight_layout()
plt.show()

## 9. Connection to the Paper

| Finding | Paper (Table 2, M=4) | Tutorial (DGP) | Pattern Match? |
|---------|---------------------|----------------|:-:|
| ATE(1-0) NTL | 0.195*** | 0.250 | Direction |
| ATE(2-1) NTL | 0.053 (n.s.) | 0.050 | Small, insignificant |
| ATE(3-1) NTL | 0.278*** | 0.300 | Large, significant |
| GATE(1-0) varies with inst. | Yes (Figs 2-3) | Yes (by design) | Upward slope |
| GATE(3-1) flat in inst. | Yes (Figs 2-3) | Yes (by design) | Flat |

### EconML vs MCF Comparison

| Feature | EconML | MCF |
|---------|:------:|:---:|
| Estimator | CausalForestDML | ModifiedCausalForest |
| Framework | DML (residualize) | Direct forest |
| Nuisance models | GBM (explicit) | Internal |
| Inference | BLB | Bootstrap |
| GATE computation | Manual | Built-in |
| Clustered SEs | No (GroupKFold) | Yes (built-in) |
| Feature importance | Built-in attr | OOB %-lost |
| CATE interpreter | Yes | No |

## 10. Conclusion

### Three Takeaways

1. **EconML's CausalForestDML is a powerful tool for heterogeneity analysis.** The DML framework provides Neyman-orthogonal estimation (robust to first-stage errors), and the causal forest captures non-linear heterogeneous effects.

2. **Institutions matter for mining, not for prices.** Stronger institutions increase the economic benefits and reduce the conflict costs of mining activities. But they do not shape the effects of mineral price changes.

3. **Different causal ML frameworks yield similar conclusions.** Both the MCF and EconML recover the same qualitative patterns from the data, providing robustness evidence.

### Exercises

1. Compare nuisance models: Replace GBM with `RandomForestRegressor`. How do results change?
2. Vary trees: Try `n_estimators=100` vs `n_estimators=1000`.
3. Remove `groups=df['district_id'].values` from `fit()`. What happens to the standard errors? Why?
4. Modify the GATE analysis: Use `quality_of_govt` quartiles instead of raw values. Do the patterns become clearer?
5. Add a new outcome: Create `y_new = ntl_log + 0.1 * conflict` and estimate effects on it.

In [ ]:
# Session info
import sys
print(f"Python: {sys.version}")
print(f"econml: {econml.__version__}")
import sklearn; print(f"scikit-learn: {sklearn.__version__}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
import matplotlib; print(f"matplotlib: {matplotlib.__version__}")